In [1]:
!pip install -q streamlit==1.67.0 altair fastapi pyarrow pytest httpx
from google.colab import drive
drive.mount('/content/drive')
!cp -r /content/drive/MyDrive/nfl-api /content/
%cd /content/nfl-api
!ls

Mounted at /content/drive
/content/nfl-api
app  data  Dockerfile  models  requirements.txt  scripts  tests


In [2]:
%%writefile app/main.py
"""FastAPI app: live win probability for any game state, plus replay of a past game."""
from functools import lru_cache
from pathlib import Path

import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

from app.model import win_probability

app = FastAPI(title="NFL Win-Probability API")
DATA = Path(__file__).resolve().parent.parent / "data" / "replay.parquet"


class GameState(BaseModel):
    home_lead: int = Field(..., ge=-80, le=80, description="home score minus away score")
    secs_left: int = Field(..., ge=0, le=4500, description="game seconds remaining (3600 at kickoff)")
    home_has_ball: bool
    yardline_100: int = Field(..., ge=1, le=99, description="yards from the OFFENSE's goal line to score")
    spread: float = Field(0.0, ge=-30, le=30, description="pre-game spread, + means home favored")


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/win-probability")
def predict(state: GameState):
    # convert to the home team's point of view, exactly like training
    home_yards = state.yardline_100 if state.home_has_ball else 100 - state.yardline_100
    p = win_probability(state.home_lead, state.secs_left, state.home_has_ball,
                        home_yards, state.spread)
    return {"home_win_prob": round(p, 4), "away_win_prob": round(1 - p, 4)}


@lru_cache
def load_plays():
    return pd.read_parquet(DATA)


@app.get("/games")
def list_games(season: int | None = None):
    plays = load_plays()
    if season is not None:
        plays = plays[plays["season"] == season]
    games = plays.drop_duplicates("game_id")[["game_id", "season", "week", "away_team", "home_team"]]
    return games.to_dict(orient="records")


@app.get("/replay/{game_id}")
def replay(game_id: str, top_swings: int = 5):
    plays = load_plays()
    g = plays[plays["game_id"] == game_id]
    if g.empty:
        raise HTTPException(status_code=404, detail=f"game {game_id} not found")

    rows = []
    for r in g.itertuples():
        has_ball = r.posteam == r.home_team
        home_yards = r.yardline_100 if has_ball else 100 - r.yardline_100
        lead = r.total_home_score - r.total_away_score
        p = win_probability(lead, r.game_seconds_remaining, has_ball, home_yards, r.spread_line)
        rows.append({"qtr": int(r.qtr), "time": r.time, "home_score": int(r.total_home_score),
                     "away_score": int(r.total_away_score), "home_win_prob": round(p, 4),
                     "nflverse_home_wp": None if pd.isna(r.vegas_home_wp) else round(float(r.vegas_home_wp), 4),
                     "desc": r.desc})

    # swing = how much this play moved the probability vs the previous play
    for i in range(len(rows)):
        prev = rows[i - 1]["home_win_prob"] if i > 0 else rows[0]["home_win_prob"]
        rows[i]["swing"] = round(rows[i]["home_win_prob"] - prev, 4)
    swings = sorted(rows, key=lambda r: abs(r["swing"]), reverse=True)[:top_swings]

    # the model doesn't know the game ended, so add a FINAL point: winner = 100%
    end = rows[-1]
    final_p = 1.0 if end["home_score"] > end["away_score"] else 0.0 if end["home_score"] < end["away_score"] else 0.5
    rows.append({**end, "time": "FINAL", "home_win_prob": final_p, "nflverse_home_wp": final_p,
                 "desc": "Game over", "swing": round(final_p - end["home_win_prob"], 4)})

    first = g.iloc[0]
    return {"game_id": game_id, "home_team": first.home_team, "away_team": first.away_team,
            "spread": float(first.spread_line), "n_plays": len(rows),
            "top_swings": swings, "plays": rows}

Overwriting app/main.py


In [3]:
!python -m pytest -q

........                                                                 [100%]
8 passed in 1.75s


In [4]:
%%writefile streamlit_app.py
"""NFL win-probability dashboard: replay any game, see the swing plays, try your own situation."""
import altair as alt
import pandas as pd
import streamlit as st

from app.main import list_games, replay
from app.model import win_probability

st.set_page_config(page_title="NFL Win Probability", page_icon="🏈", layout="wide")
st.title("🏈 NFL Win-Probability Engine")
st.caption("Logistic regression built from scratch on 300K+ NFL plays (2015–2022), "
           "tested on 2023–2025. Compared against nflverse's own model.")

tab_replay, tab_whatif, tab_model = st.tabs(["Replay a game", "What-if", "How good is it?"])

# ---------------- Replay ----------------
with tab_replay:
    games = pd.DataFrame(list_games())
    games["label"] = games.apply(
        lambda g: f"{g.season} wk {g.week}: {g.away_team} @ {g.home_team}", axis=1)
    games = games.sort_values(["season", "week"], ascending=[False, True])
    sb51 = "2016_21_NE_ATL"
    games.loc[games.game_id == sb51, "label"] = "⭐ Super Bowl LI: NE @ ATL (28–3 comeback)"
    options = [sb51] + [g for g in games.game_id if g != sb51]
    labels = dict(zip(games.game_id, games.label))

    game_id = st.selectbox("Pick a game", options, format_func=lambda g: labels[g])
    r = replay(game_id, top_swings=5)
    home, away = r["home_team"], r["away_team"]
    plays = pd.DataFrame(r["plays"])
    plays["play"] = range(1, len(plays) + 1)

    side = st.radio("Show win probability for", [home, away], horizontal=True,
                    index=1 if game_id == sb51 else 0)   # SB LI: show the Patriots' comeback
    flip = side == away
    plays["My model"] = 1 - plays["home_win_prob"] if flip else plays["home_win_prob"]
    plays["nflverse"] = (1 - plays["nflverse_home_wp"]) if flip else plays["nflverse_home_wp"]

    last = plays.iloc[-1]
    c1, c2, c3 = st.columns(3)
    c1.metric("Final score", f"{away} {last.away_score} – {last.home_score} {home}")
    c2.metric(f"Lowest {side} win prob", f"{plays['My model'].min():.1%}")
    c3.metric(f"Highest {side} win prob", f"{plays['My model'].max():.1%}")

    long = plays.melt(id_vars=["play", "qtr", "time", "away_score", "home_score", "desc"],
                      value_vars=["My model", "nflverse"], var_name="model", value_name="p")
    chart = alt.Chart(long).mark_line(strokeWidth=2).encode(
        x=alt.X("play:Q", title="play number"),
        y=alt.Y("p:Q", title=f"P({side} wins)", scale=alt.Scale(domain=[0, 1]),
                axis=alt.Axis(format="%")),
        color=alt.Color("model:N", scale=alt.Scale(domain=["My model", "nflverse"],
                                                   range=["#2a78d6", "#eb6834"]),
                        legend=alt.Legend(orient="top", title=None)),
        tooltip=[alt.Tooltip("model:N"), alt.Tooltip("p:Q", format=".1%", title="win prob"),
                 "qtr:Q", "time:N", "away_score:Q", "home_score:Q", "desc:N"],
    )
    half = alt.Chart(pd.DataFrame({"y": [0.5]})).mark_rule(color="#999999", strokeDash=[4, 4]).encode(y="y:Q")
    st.altair_chart((half + chart).properties(height=380).interactive(bind_y=False),
                    width="stretch")

    st.subheader("Top 5 swing plays")
    sw = pd.DataFrame(r["top_swings"])
    sw["swing"] = -sw["swing"] if flip else sw["swing"]
    sw["score"] = sw.apply(lambda s: f"{away} {s.away_score} – {s.home_score} {home}", axis=1)
    sw["swing"] = sw["swing"].map(lambda s: f"{s:+.0%}")
    st.dataframe(sw[["qtr", "time", "score", "swing", "desc"]], hide_index=True,
                 width="stretch")

# ---------------- What-if ----------------
with tab_whatif:
    st.write("Set up any game situation and see the home team's chance.")
    a, b = st.columns(2)
    lead = a.slider("Home lead (points)", -28, 28, 0)
    minutes = a.slider("Minutes left in the game", 0.0, 60.0, 30.0, step=0.5)
    has_ball = b.radio("Who has the ball?", ["Home", "Away"], horizontal=True) == "Home"
    yards = b.slider("Yards the offense needs to score", 1, 99, 75)
    spread = b.slider("Pre-game spread (+ = home favored)", -14.0, 14.0, 0.0, step=0.5)
    home_yards = yards if has_ball else 100 - yards
    p = win_probability(lead, minutes * 60, has_ball, home_yards, spread)
    st.metric("Home win probability", f"{p:.1%}")
    st.progress(p)

# ---------------- Model card ----------------
with tab_model:
    st.write("Test seasons 2023–2025, 130,811 plays. Lower log loss and Brier are better.")
    st.dataframe(pd.DataFrame({
        "model": ["Always guess the average", "nflverse home_wp (no spread)", "My logistic regression",
                  "My XGBoost (monotonic)", "nflverse vegas_home_wp"],
        "log loss": [0.6898, 0.4814, 0.4541, 0.4618, 0.4486],
        "Brier": [0.2483, 0.1627, 0.1503, 0.1523, 0.1486],
    }), hide_index=True, width="stretch")
    st.markdown(
        "- **Features:** home lead, time left, possession, field position, pre-game spread, "
        "and two interactions (lead × time elapsed, spread × time left).\n"
        "- **Calibration:** expected calibration error ≈ 0.01. When it says 70%, the home team wins about 70%.\n"
        "- **Why not XGBoost?** 300K plays but only ~1,800 independent games; trees overfit "
        "game-level noise. The simpler model generalized better.")

Writing streamlit_app.py


In [6]:
from streamlit.testing.v1 import AppTest
at = AppTest.from_file("/content/nfl-api/streamlit_app.py", default_timeout=60).run()
print("errors:", list(at.exception))
print([(m.label, m.value) for m in at.metric])

2026-11-07 22:17:31.733 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


errors: []
[('Final score', 'NE 34 – 28 ATL'), ('Lowest NE win prob', '1.2%'), ('Highest NE win prob', '100.0%'), ('Home win probability', '49.2%')]


In [7]:
%%writefile requirements.txt
fastapi
uvicorn[standard]
pandas
pyarrow
numpy
pytest
httpx
streamlit==1.67.0
altair

Overwriting requirements.txt


In [8]:
%%writefile Dockerfile
FROM python:3.11-slim
WORKDIR /code
ENV HOME=/tmp
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app/ app/
COPY models/ models/
COPY data/ data/
COPY streamlit_app.py .
EXPOSE 8501
CMD ["streamlit", "run", "streamlit_app.py", "--server.port=8501", "--server.address=0.0.0.0"]

Overwriting Dockerfile


In [9]:
!rm -rf .pytest_cache app/__pycache__ tests/__pycache__
!cp -r /content/nfl-api /content/drive/MyDrive/
!cd /content && rm -f nfl-api.zip && zip -rq nfl-api.zip nfl-api
from google.colab import files
files.download("/content/nfl-api.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>